# 05. Doc2Vecモデルの学習

04で作成したトークンCSVから、**銘柄ごとの味わいベクトル**を学習します（入力: `reviews_tokenized.csv` → 出力: Doc2Vecモデル）。
前処理は済んでいるため、このノートブックには形態素解析のコードはありません。

- 口コミ1件 = 1ドキュメント、タグ = 銘柄（都道府県_銘柄_蔵元）
- 最終設定: `vector_size=75, window=5, min_count=2, epochs=20, dm=1`
- `seed=42, workers=1` で再現性を確保

In [ ]:
# 依存ライブラリのインストール（Colab）
!pip install -q gensim scikit-learn pandas numpy scipy

## 設定

In [ ]:
# --- 実行環境の設定 ---------------------------------------------------------
# Colab: Google Driveをマウントし、BASE_DIR 配下を読み書きします。
# ローカル: 環境変数 SAKE_PROJECT_DIR でデータ置き場を指定できます。
import os, sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/作成したプログラム等/日本酒口コミ分析'  # 自分のDriveのパスに変更
    # sake_nlp.py をDriveの BASE_DIR 直下に置くか、Colabの左ペインにアップロードしてください
    sys.path.append(BASE_DIR)
except ImportError:
    BASE_DIR = os.environ.get('SAKE_PROJECT_DIR', './sake_review_analysis')

CSV_DIR = f'{BASE_DIR}/口コミ_CSV'          # 収集・加工したCSV
MODEL_DIR = f'{BASE_DIR}/機械学習モデル'      # 学習済みDoc2Vecモデル
for d in (CSV_DIR, MODEL_DIR):
    os.makedirs(d, exist_ok=True)

REVIEWS_CSV = f'{CSV_DIR}/reviews_with_water_hardness.csv'   # 03で作成した口コミ+硬度
TOKENIZED_CSV = f'{CSV_DIR}/reviews_tokenized.csv'           # 04の出力 = 05, 06の入力
MODEL_PATH = f'{MODEL_DIR}/final_masked_sudachi_doc2vec.model'  # 05の出力 = 06の入力


## 1. トークンCSVの読み込み

In [ ]:
import numpy as np
import pandas as pd
from gensim.models.doc2vec import Doc2Vec
import sake_nlp as nlp

df = nlp.load_tokenized(TOKENIZED_CSV)
docs = nlp.to_tagged_docs(df)
print(f'ドキュメント数: {len(docs)} / 銘柄数: {df["tag"].nunique()}')

## 2. 最終モデルの学習

In [ ]:
PARAMS = dict(vector_size=75, window=5, min_count=2, epochs=20, dm=1, seed=42, workers=1)

model = Doc2Vec(**PARAMS)
model.build_vocab(docs)
print(f'語彙数: {len(model.wv)} / 銘柄数: {len(model.dv)}')

model.train(docs, total_examples=model.corpus_count, epochs=model.epochs)
model.save(MODEL_PATH)
print(f'✅ 保存: {MODEL_PATH}')

## 3. 動作確認：学習した語の意味が近いか

Doc2Vecと同時に学習された単語ベクトルで、味わい語の近傍を確認します。

In [ ]:
for w in ['フルーティー', '辛口', '酸味']:
    if w in model.wv:
        print(w, '→', [f'{x}({s:.2f})' for x, s in model.wv.most_similar(w, topn=5)])

## 4. ハイパーパラメータの検討（任意・時間がかかります）

`vector_size × epochs` を振り、クラスタリング品質（シルエット係数 等）と
**「口コミ件数でクラスタが決まっていないか」**（件数との効果量 η²）を確認します。

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy import stats

RUN_GRID_SEARCH = False   # True にすると実行
K = 4
review_counts = df['tag'].value_counts().to_dict()

if RUN_GRID_SEARCH:
    rows = []
    for vs in [30, 50, 75, 100]:
        for ep in [10, 20, 40]:
            m = Doc2Vec(vector_size=vs, window=5, min_count=2, epochs=ep, dm=1, seed=42, workers=1)
            m.build_vocab(docs)
            m.train(docs, total_examples=m.corpus_count, epochs=m.epochs)
            tags = list(m.dv.index_to_key)
            X = np.array([m.dv[t] for t in tags])
            labels = KMeans(n_clusters=K, random_state=42, n_init=10).fit_predict(X)
            counts = np.array([review_counts[t] for t in tags])
            f, _ = stats.f_oneway(*[counts[labels == c] for c in range(K)])
            eta2 = f * (K - 1) / (f * (K - 1) + len(tags) - K)
            rows.append({'vector_size': vs, 'epochs': ep,
                         'silhouette': silhouette_score(X, labels),
                         'davies_bouldin': davies_bouldin_score(X, labels),
                         'pca_variance': PCA(2, random_state=42).fit(X).explained_variance_ratio_.sum(),
                         'review_count_eta2': eta2})
    df_grid = pd.DataFrame(rows).sort_values('silhouette', ascending=False)
    print(df_grid.to_string(index=False))